In [ ]:
# librerias de kaggle no funcionan, por esto borramos y descargamos de nuevo:
!pip uninstall -y kaggle kagglesdk

# Re-instalarla
!pip install -q kaggle kagglesdk

# REINICIAR SESION!!!!!



# 1.   Clasificacion de imagenes de fondo de ojo para deteccion de retinopatia diabetica
Mi TFM consiste en desarrollar un sistema de clasificacion de imagenes de fondos de ojo proporcionadas por la Sociedad Española de Diabetes para detectar retinopatia diabetica.
La retinopatia diabetica es una complicacion de la Diabetes Mellitus, y pasa cuando los vasos sanguineos del tejido sensible a la luz se dañan y, a veces, puede conducir a la ceguera. (García-Medina & Romo-García, 2023) (Boyd, 2019)

#2.   Problema y dataset (origen, variables, target, tamaño, ejemplos).
Para esta tarea se usará el dataset APTOS 2019, disponible en Kaggle, mentras que la Sociedad Española de Diabetes nos da permiso para usar su proprio dataset.


In [ ]:
import os
import json
from google.colab import userdata

# variables de entorno para usar Kaggle
os.environ['KAGGLE_USERNAME'] = "ValeValeM"
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

# Crear config file
!mkdir -p /root/.config/kaggle
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump({"username": os.environ['KAGGLE_USERNAME'], "kaggle_api_token": os.environ['KAGGLE_API_TOKEN']}, f)
!chmod 600 /root/.config/kaggle/kaggle.json

# Descarga de los datos
!kaggle competitions download -c aptos2019-blindness-detection --force

In [ ]:
# unzip la descarga
!unzip -q aptos2019-blindness-detection.zip

#3.   Exploración y calidad del dato (EDA ligera, faltantes, clases desbalanceadas, outliers).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

In [ ]:
train_df = pd.read_csv("train.csv") # nombre de la imagen y diagnostico de un medico
test_df = pd.read_csv("test.csv") # solo lista de imagenes de test
train_df, test_df

In [ ]:
train_df["diagnosis"].unique() # las etiquetas del train
len(train_df) # numero de imagenes de train
len(test_df) # numero de imagenes de test

Desde la pagina web del Kaggle, esto es lo que significan las etiquetas:
*   0 - No DR
*   1 - Mild
*   2 - Moderate
*   3 - Severe
*   4 - Proliferative DR

In [ ]:
class_names = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
len(class_names)

In [ ]:
from PIL import Image
import random

# visulizo la primera imagen
ruta = 'train_images/' # ruta de las imagenes de entrenamiento

img_name = os.listdir(ruta)[0] # primera imagen
ruta = os.path.join(ruta, img_name)
plt.figure()
plt.imshow(Image.open(ruta))
plt.title(f"{img_name}")
plt.show()

In [ ]:
# voy a ver las dimensiones de esta imagen
with Image.open(ruta) as img:
    print(img.size)

In [ ]:
# voy a ver cual es el diagnostico de esta imagen
img_name = img_name.replace('.png', '')
train_df.loc[train_df['id_code'] == img_name]


In [ ]:
# la imagen anterior dice diagnostico 0, que es ojo sin DR
# voy a ver algunas fotos de todas las clases para ver que las distingue

# numero de imagenes para cada categoria
data_stats = []
for i in range(len(class_names)):
  count = len(train_df.loc[train_df['diagnosis'] == i])
  data_stats.append({
      'Category': class_names[i],
      'Count': count
      # 'Sample Resolution': f"{width}x{height}"
  })
df_stats = pd.DataFrame(data_stats)

print(df_stats)
""" Noto un desbalanzeo de las clases, de hecho las imagenes sin enfermedades
son x2 - x10 veces mas que las con problemas. Esto es comprendible porque
el examen se hace de manera preventiva, para detectar la enfermedad antes de que
crea sintomas en la persona, entonces mucha gente lo hace y no tiene DR"""

In [ ]:
# Mostrar cuatro imagenes random para cada categoria
plt.figure(figsize=(9, 12))
path = 'train_images/'
n_images = 4 # 4 imagenes aleatorias para cada categoria para verlas

for i in range(len(class_names)):
  img_names = train_df.loc[train_df['diagnosis'] == i]['id_code'].values
  img_names = random.sample(list(img_names), n_images)

  for j in range(n_images):
        img_name = img_names[j]
        img_path = os.path.join(path, img_name + '.png')


        indice_plot = i * n_images + j + 1
        plt.subplot(len(class_names), n_images, indice_plot)

        plt.imshow(Image.open(img_path))
        # plt.axis('off')
        plt.title(class_names[i])

plt.tight_layout()
plt.show()

Tambien veo que las imagenes tienen diferente tamaño. Al principio corrigió esto en el generador de datos para keras, pero hacía el modelo muy lento (3s/step), entonces las redimensionamos ahora:

In [ ]:
""" Redimensiono las imagenes antes de ponerlas en el generador porque hay
muchas muy grandes (~2000 x 2000 pixeles) entonces redimensionarlas en el
generador lleva mucho tiempo porque lo hace una a una."""

import cv2

source_path = '/content/train_images/'
target_path = '/content/train_224/'
os.makedirs(target_path, exist_ok=True)

size = (224, 224) # tamaño comun para CNNs

print("Redimensionando imágenes...")
print("Esto lleva algunos minutos")

for img_name in os.listdir(source_path):
  img = cv2.imread(os.path.join(source_path, img_name))
  if img is not None:
    img_resized = cv2.resize(img, size, interpolation=cv2.INTER_AREA) # Redimensionar
    cv2.imwrite(os.path.join(target_path, img_name), img_resized) # Guardar en la nueva carpeta
print("imagenes redimensionadas")

#4.   Pre-procesamiento del dato

In [ ]:
import matplotlib.image as mpimg
# miro al valor de los pixeles de la segunda imagen
ruta = '/content/train_224/' # ruta de las imagenes de entrenamiento

img_name = os.listdir(ruta)[1] # segunda imagen
ruta = os.path.join(ruta, img_name)
img = mpimg.imread(ruta)

plt.figure(figsize=(5, 5))
plt.imshow(Image.open(ruta))
plt.title(f"{img_name}")
plt.xlim(0, 224)
plt.ylim(0, 224)
plt.show()

print("dimensiones de la foto:", np.shape(img))
print("Tipo de dato de la foto:", img.dtype) # las imagenes tiene tipo de datos 'float32' entonces serán normalizados ya
print("valor minimo de los pixeles:", img.min()) # valor minimo de los pixeles
print("valor maximo de los pixeles", img.max()) # valor maximo de los pixeles es cerca de 0.95 entonces las fotos no necesitan normalizacion


Veo que la foto ya esta normalizada (tiene valores entre 0 y 1)

#5.   Baseline: CNN simple. (pérdida, métricas, early stopping, regularización, etc.).

## Estructura del modelo baseline

In [ ]:
from tensorflow.keras import layers, models

model_baseline = models.Sequential([
    # Input sin resizing porque lo hacemos cuando se definen los datos de train y test (antes de entrenar el modelo)
    layers.Input(shape=(224, 224, 3)), # dimensiones estandar para inputs deCNN

    # data augmentation para enseñar al modelo que las plantas tambien pueden ser vistas de otras prospectivas
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),

    # convoluciones
    layers.Conv2D(32, (3, 3), activation='relu', padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu', padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(256, (3, 3), activation='relu', padding="same"),

    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # se transforma en una fila sola, pero no añade tantos parametros cuanto el Flatten
    layers.GlobalAveragePooling2D(),

    # capa densa
    layers.Dense(128, activation="relu"),
    # capa dropout
    layers.Dropout(0.5),

    # capa softmax de salida con 12 outputs
    layers.Dense(5, activation='softmax')
])

In [ ]:
model_baseline.summary()

In [ ]:
# COMPILE:
model_baseline.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy", # categorical porque haremos one hot encoding
    metrics = ["accuracy"]
)

## Entrenamiento del baseline

Preparar los datos para usarlos en el modelo

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BATCH_SIZE = 64 # batches medio/pequeños

train_df = pd.read_csv('train.csv')
train_df['id_code'] = train_df['id_code'].apply(lambda x: f"{x}.png") # añadir la extension del fichero
train_df['diagnosis'] = train_df['diagnosis'].astype(str) # Convertir la columna de diagnóstico a string (importante para clasificación)


In [ ]:
datagen = ImageDataGenerator(
    validation_split=0.2 # 20% de las imagenes utilizadas como validacion
)

# Generador de ENTRENAMIENTO
train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/content/train_224', # Ruta de las imagenes redimensionadas
    x_col="id_code",
    y_col="diagnosis",
    target_size=(224, 224),           # mismo de las dimensiones en la carpeta de imagenes redimensionadas
    batch_size=BATCH_SIZE,
    class_mode="categorical",         # Genera One-Hot encoding automáticamente
    subset="training",
    shuffle=True,
    seed=123
)

# Generador de VALIDACIÓN
val_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/content/train_224',
    x_col="id_code",
    y_col="diagnosis",
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

Callbacks implementados:

In [ ]:
import tensorflow as tf

# Implementamos early stopping:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5, # si despues 5 epocas val_loss no mejora, stop
    restore_best_weights=True # deja los mejores pesos como resultados finales
)

# reducir learning rate cuando se queda plano
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,        # multiplica el learning rate por 0.2 (mas pequeño)
    patience=3,        # espera 3 epocas de no mejorias antes de agir
    min_lr=1e-6,       # no lo deja ir más lento que así
    verbose=1          # verbose
)

# guardar checkpoint solo si accuracy ha mejorado
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_DR_model.keras',
    monitor='val_accuracy',
    save_best_only=True, # solo guarda si accuracy ha mejorado
    mode='max',
    verbose=1
)

Equilibrar el desbalanceo de clases

In [ ]:
# Equilibro el desbalanceo de clases (hay mas sanos que enfermos)
weights = {0: 1.0, 1: 2.5, 2: 1.5, 3: 3.0, 4: 2.5}

In [ ]:
# entrenamiento del modelo
history_baseline = model_baseline.fit(
    train_generator,
    class_weight=weights,
    validation_data=val_generator,
    epochs = 20, # un poco mas de epocas porque tenemos el early_stop y reduce_lr si no mejora
    callbacks=[early_stop, reduce_lr, checkpoint]
)

## Validación del baseline

In [ ]:
# Validacion
test_loss, test_acc = model_baseline.evaluate(val_generator)


In [ ]:
print(f"Accuracy de la evaluacion:", test_acc)

#7.   Transfer Learning Modelos DL (arquitecturas probadas + por qué).

Por recomendacion de Gemini descargamos el modelo EfficientNetB0 para hacer transfer learning, sabendo que las diferencias entre ojos con o sin DR son bastante sutiles y una CNN basica no puede dar una accuracy muy alta.

## Estructura

In [ ]:
from tensorflow.keras.applications import EfficientNetB0 # red bastante ligera

# Cargo el modelo base sin la última capa
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False, # quita las capas de clasificación de ImageNet (las 1000 clases originales)
    input_shape=(224, 224, 3)
)

base_model.trainable = False # Congelo el modelo base

# Añado mis propias capas arriba
model_TL = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(), # Convierte en churro
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

model_TL.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model_TL.summary()

## Entrenamiento

In [ ]:
""" Escribo de nuevo los generadores para que tengan input entre 1 y 255 como
quiere EfficientNetB0"""

from tensorflow.keras.applications.efficientnet import preprocess_input

datagen = ImageDataGenerator(
    validation_split=0.2, # 20% de las imagenes utilizadas como validacion
    preprocessing_function=preprocess_input # EfficientNetB0 quiere input de 1 a 255, no entre 0 y 1
)

# Generador de ENTRENAMIENTO
train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/content/train_224', # Ruta de las imagenes redimensionadas
    x_col="id_code",
    y_col="diagnosis",
    target_size=(224, 224),           # mismo de las dimensiones en la carpeta de imagenes redimensionadas
    batch_size=BATCH_SIZE,
    class_mode="categorical",         # Genera One-Hot encoding automáticamente
    subset="training",
    shuffle=True,
    seed=123
)

# Generador de VALIDACIÓN
val_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/content/train_224',
    x_col="id_code",
    y_col="diagnosis",
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

In [ ]:
# Nuevo checkpoint para no sobre-escribir los viejos si se van empeorando
checkpoint_TL = tf.keras.callbacks.ModelCheckpoint(
    'best_finetuning.keras',
    monitor='val_accuracy',
    save_best_only=True, # solo guarda si accuracy ha mejorado
    mode='max',
    verbose=1
)

# Implementamos early stopping:
early_stop_TL = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True # deja los mejores pesos como resultados finales
)

# reducir lr cuando se queda plano
reduce_lr_TL = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,        # multiplica el learning rate por 0.2 (mas pequeño)
    patience=3,        # espera 3 epocas de no mejorias antes de agir
    min_lr=1e-7,       # no lo deja ir más lento que así
    verbose=1          # verbose
)

In [ ]:
# Calentamento del modelo de TL con pocas epocas
history = model_TL.fit(
    train_generator,
    class_weight=weights,
    validation_data=val_generator,
    epochs = 5,
    callbacks=[early_stop_TL, reduce_lr_TL, checkpoint_TL]
)

In [ ]:
""" Aumento la paciencia de reduce_lr durante la fase de fine-tuning para ver si
necesita mas tentavios para encontrar learning rate adecuado """

reduce_lr_ft = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=4, # aumento desde 3 a 4 de paciencia
    min_lr=1e-7,
    verbose=1
)

In [ ]:
# Fine-tuning del model_TL
base_model.trainable = True # desbloqueo toda la red

# model_TL = models.Sequential([
#     base_model,
#     layers.GlobalAveragePooling2D(), # Convierte en churro
#     layers.BatchNormalization(),
#     layers.Dropout(0.3),
#     layers.Dense(512, activation='relu'),
#     layers.Dropout(0.5),
#     layers.Dense(5, activation='softmax')
# ])

model_TL.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6), # LR muy bajo
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model_TL.summary()

In [ ]:
# Entrenar por 10-15 epocas mas
history_TL = model_TL.fit(
    train_generator,
    class_weight=weights,
    validation_data=val_generator,
    epochs = 15,
    callbacks=[early_stop_TL, reduce_lr_ft, checkpoint_TL]
)

## Validacion

In [ ]:
# Validacion
test_loss, test_acc = model_TL.evaluate(val_generator)


In [ ]:
print(f"Accuracy de la evaluacion:", test_acc)

#8.   Resultados y comparación (tablas, curvas, métricas, coste computacional).

## Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

# def plot_compare(history_baseline, history_TL, metric='accuracy'):
plt.figure(figsize=(12, 6))
# CNN Simple
plt.plot(history_baseline.history['accuracy'], label='CNN Train', color='blue', linestyle='--')
plt.plot(history_baseline.history['val_'+'accuracy'], label='CNN Val', color='blue')

# EfficientNet (Fase 1 + Fase 2 concatenadas)
plt.plot(history_TL.history['accuracy'], label='EffNet Train', color='red', linestyle='--')
plt.plot(history_TL.history['val_'+'accuracy'], label='EffNet Val', color='red')

plt.title(f'Comparación de {'accuracy'}')
plt.xlabel('Epochs')
plt.ylabel('accuracy')
plt.legend()
plt.show()

Las curvas de validacion saltan un poco, que puede indicar un learning rate demasiado alto.
Tambien la CNN, siendo mas simple, aparece mas robusta, mentras que la EfficientNet tiene el potencial de llegar mas lejos en terminos de accuracy pero es mas sensible a pequeñas variaciones de parametros y es mas dificil de tunear.
Tambien la CNN tiene accuracy de train mas alta que en el test, que puede ser sintoma de overfitting, mientras que la EfficientNet no.


## Matriz de confusion

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Predicciones con CNN
y_pred_CNN = model_baseline.predict(val_generator)
y_pred_CNN = np.argmax(y_pred_CNN, axis=1)
y_true = val_generator.classes

cm = confusion_matrix(y_true, y_pred_CNN)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

se puede ver que la clase 2 (Mild DR) está mal clasificada muchas veces, y las clases 3 y 4 (advanced DR y proliferative DR) han sido siempre mal clasificadas, esto puede ser causado por el desbalanzeo de las clases, donde el dataset no tiene bastante imagenes de estas clases para aprender a clasificarlas.

In [ ]:
# Predicciones con EfficientNet
y_pred_TL = model_TL.predict(val_generator)
y_pred_TL = np.argmax(y_pred_TL, axis=1)
y_true = val_generator.classes

cm = confusion_matrix(y_true, y_pred_TL)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

Se puede ver que el modelo sabe distinguir bien la clase 0 (ojo sin DR) pero todavia tiene dificultades con las otras, que está en linea con la baja accuracy obtenida (0.73 es el mejor valor que obtuve en el EfficientNet)

#11.   Conclusiones y trabajo futuro (qué funcionó, qué no, y próximos pasos).

El modelo de Transfer Learning con EfficientNet todavia no ha lelgado a su maximo potencial y tiene que ser mejorado todavia.
El modelo tambien pudiera aprender a enfocarse en el fondo de ojo y no efocarse el los bordes negros, o se puede añadir al dataset otro dataset (por ejemplo EYEpacs) para que los modelos puedan aprender con más imagenes y balancear las clases un poquito más.
El codigo tambien tarda mucho a ejecutarse porque algunas imagenes son muy grandes (tambien ~2000x2000 pixeles) entonces lleva mucho tiempo a pre-procesarlas.

# 12) Referencias:
1.   García-Medina, K. A., & Romo-García, E. (2023). Diagnóstico y clasificación de la retinopatía diabética utilizando imágenes de fondo de ojo de campo ultra amplio, comparando los sistemas Optos® y Clarus 700®. REVMEDUAS, 13(1), 33–44. https://doi.org/10.28960/revmeduas.2007-8013.v13.n1.005

2.   Boyd, K. (2019). What Is Diabetic Retinopathy? American Academy of Ophthalmology. https://www.aao.org/eye-health/diseases/what-is-diabetic-retinopathy

3.   ahmadelsallab. (2025). GitHub - ahmadelsallab/APTOS: Diabetic
Retinopathy using Computer Vision and ConvNets. GitHub. https://github.com/ahmadelsallab/APTOS

‌

